# Module 3 — SOLUTION: The Full RAG Pipeline

> **Instructor note:** This is the complete solution. Share after the exercise session.

In [ ]:
import json
import os

from openai import OpenAI
import chromadb
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

load_dotenv("../.env")

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.PersistentClient(path="../chroma_db")
collection = chroma_client.get_or_create_collection(
    name="workshop_rag", metadata={"hnsw:space": "cosine"}
)
client = OpenAI(
    base_url=os.getenv("NETLIGHT_BASE_URL", "https://llm.netlight.com/v1"),
    api_key=os.getenv("NETLIGHT_API_KEY"),
)
FAST_MODEL = os.getenv("NETLIGHT_MODEL_FAST", "claude-haiku-4-5")
SMART_MODEL = os.getenv("NETLIGHT_MODEL_SMART", "claude-sonnet-4-5")

print(f"Collection has {collection.count()} docs")

In [ ]:
# ── Solution 3.1: Improved system prompt ──────────────────────────────────
SYSTEM_PROMPT_V2 = """\
You are a knowledgeable insurance and technology assistant.

Rules:
1. Answer ONLY using information from the numbered context chunks provided.
2. Cite sources using the chunk number, e.g. "According to [1]..."
3. If the context is insufficient, say exactly: "The provided context does not contain enough information to answer this question."
4. Keep answers concise and structured. Use bullet points for lists.
5. If multiple chunks contain conflicting information, note the discrepancy.
6. Never fabricate facts or use outside knowledge."""


# ── Solution 3.2: Improved user prompt ────────────────────────────────────
def build_user_prompt_v2(question: str, context_chunks: list[dict]) -> str:
    lines = []
    for i, chunk in enumerate(context_chunks, 1):
        sim = chunk.get("similarity", 0)
        lines.append(f"[{i}] Source: {chunk['source']} (relevance: {sim:.2f})")
        lines.append(chunk["text"])
        lines.append("")
    context_text = "\n".join(lines)

    return f"""Retrieved Context:
{context_text}
Question: {question}

Answer:"""


# Quick test
q = "What are DKV Belgium's General Terms and Conditions?"
chunks = [
    {"text": c["text"], "source": c["source"], "similarity": c["similarity"]}
    for c in (
        lambda q_emb: [
            {"text": t, "source": m["source"], "similarity": 1 - d}
            for t, m, d in zip(
                collection.query(query_embeddings=[q_emb], n_results=4)["documents"][0],
                collection.query(query_embeddings=[q_emb], n_results=4)["metadatas"][0],
                collection.query(query_embeddings=[q_emb], n_results=4)["distances"][0],
            )
        ]
    )(embed_model.encode(q).tolist())
]

prompt = build_user_prompt_v2(q, chunks)
response = client.chat.completions.create(
            model=FAST_MODEL,
            max_tokens=512,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT_V2},
                {"role": "user", "content": prompt},
            ],
        )
print(response.choices[0].message.content)

In [ ]:
# ── Solution 3.3: RAGPipeline class ───────────────────────────────────────
class RAGPipeline:
    def __init__(
        self,
        collection,
        embed_model,
        llm_client: OpenAI,
        model: str = FAST_MODEL,
        top_k: int = 5,
        system_prompt: str = SYSTEM_PROMPT_V2,
        min_similarity: float = 0.0,
    ):
        self.collection = collection
        self.embed_model = embed_model
        self.llm_client = llm_client
        self.model = model
        self.top_k = top_k
        self.system_prompt = system_prompt
        self.min_similarity = min_similarity

    def retrieve(self, question: str) -> list[dict]:
        q_emb = self.embed_model.encode(question).tolist()
        results = self.collection.query(
            query_embeddings=[q_emb],
            n_results=self.top_k,
            include=["documents", "metadatas", "distances"],
        )
        chunks = [
            {"text": text, "source": meta["source"], "similarity": 1 - dist}
            for text, meta, dist in zip(
                results["documents"][0],
                results["metadatas"][0],
                results["distances"][0],
            )
        ]
        return [c for c in chunks if c["similarity"] >= self.min_similarity]

    def build_prompt(self, question: str, chunks: list[dict]) -> str:
        return build_user_prompt_v2(question, chunks)

    def ask(self, question: str) -> dict:
        chunks = self.retrieve(question)
        prompt = self.build_prompt(question, chunks)

        response = self.llm_client.chat.completions.create(
            model=self.model,
            max_tokens=1024,
            messages=[
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": prompt},
            ],
        )
        return {
            "question": question,
            "answer": response.choices[0].message.content,
            "contexts": [c["text"] for c in chunks],
            "sources": [c["source"] for c in chunks],
        }


rag = RAGPipeline(
    collection=collection,
    embed_model=embed_model,
    llm_client=client,
    model=FAST_MODEL,
    top_k=5,
    system_prompt=SYSTEM_PROMPT_V2,
)

result = rag.ask("What are DKV Belgium's General Terms and Conditions?")
print(f"Q: {result['question']}\n")
print(f"A: {result['answer']}\n")
print(f"Sources: {list(set(result['sources']))}")

In [ ]:
# Run full test set and save for Module 4 evaluation
with open("../data/sample_qa.json") as f:
    qa_data = json.load(f)

print("Running RAG on all test questions...")
all_results = []
for qa in qa_data["questions"]:
    result = rag.ask(qa["question"])
    result["ground_truth"] = qa["ground_truth"]
    result["id"] = qa["id"]
    all_results.append(result)
    print(f"  ✓ {qa['id']}: {qa['question'][:50]}...")

# Save for Module 4
import json
with open("../data/rag_results.json", "w") as f:
    json.dump(all_results, f, indent=2)

print(f"\nSaved {len(all_results)} results to ../data/rag_results.json")

In [ ]:
# ── Solution 3.4: Comparison experiment ───────────────────────────────────
rag_v1 = RAGPipeline(collection=collection, embed_model=embed_model, llm_client=client,
                     model=FAST_MODEL, top_k=3, system_prompt=SYSTEM_PROMPT_V2)

rag_v2 = RAGPipeline(collection=collection, embed_model=embed_model, llm_client=client,
                     model=FAST_MODEL, top_k=7, system_prompt=SYSTEM_PROMPT_V2)

test_q = "What are the main types of insurance and how do they differ?"

r1 = rag_v1.ask(test_q)
r2 = rag_v2.ask(test_q)

print("=== top_k=3 ===")
print(f"Sources: {list(set(r1['sources']))}")
print(r1["answer"])

print("\n=== top_k=7 ===")
print(f"Sources: {list(set(r2['sources']))}")
print(r2["answer"])